In [1]:
# =========================
# 🔹 INSTALL LIBRARIES
# =========================
!pip install vncorenlp transformers torch scikit-learn tqdm

# =========================
# 🔹 DOWNLOAD VnCoreNLP
# =========================
import os

vncorenlp_dir = "VnCoreNLP-master"
vncorenlp_zip = "master.zip"

if not os.path.exists(vncorenlp_dir):
    print(f"Downloading {vncorenlp_zip}...")
    !wget -q https://github.com/vncorenlp/VnCoreNLP/archive/refs/heads/{vncorenlp_zip}
    print(f"Unzipping {vncorenlp_zip}...")
    !unzip -q {vncorenlp_zip}
    print("VnCoreNLP download and extraction complete.")
else:
    print("VnCoreNLP already exists. Skipping download and extraction.")

# =========================
# 🔹 DOWNLOAD DATASET (VNTC)
# =========================
vntc_dir = "VNTC"
if not os.path.exists(vntc_dir):
    print(f"Cloning {vntc_dir} repository...")
    !git clone https://github.com/duyvuleo/VNTC.git
    print("VNTC repository cloned.")
else:
    print("VNTC repository already exists. Skipping cloning.")

# =========================
# 🔹 UNRAR DATASET
# =========================
# Install unrar if not already present
!apt-get update -qq > /dev/null
!apt-get install -y unrar > /dev/null
print("Unrar installed.")

train_full_dir = "/content/VNTC/Data/10Topics/Ver1.1/Train_Full"
test_full_dir = "/content/VNTC/Data/10Topics/Ver1.1/Test_Full"

if not os.path.exists(train_full_dir) or not os.path.exists(test_full_dir):
    print("Extracting dataset archives...")
    !unrar x /content/VNTC/Data/10Topics/Ver1.1/Train_Full.rar /content/VNTC/Data/10Topics/Ver1.1/
    !unrar x /content/VNTC/Data/10Topics/Ver1.1/Test_Full.rar /content/VNTC/Data/10Topics/Ver1.1/
    print("Dataset extraction complete.")
else:
    print("Dataset already extracted. Skipping extraction.")

Streaming output truncated to the last 5000 lines.
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6210).txt      89%  OK 
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6079).txt      89%  OK 
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6083).txt      89%  OK 
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6086).txt      89%  OK 
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6088).txt      89%  OK 
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6312).txt      89%  OK 
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6092).txt      89%  OK 
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6094).txt      89%  OK 
Extracting  /content/VNTC/Data/10Topics/Ver1.1/Test_Full/Van hoa/VH_VNE_T_ (6

In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from vncorenlp import VnCoreNLP
from collections import Counter
import random
import numpy as np
import pickle
from tqdm import tqdm
import time
# =========================
# 🔹 CONFIG
# =========================
TRAIN_DIR = "/content/VNTC/Data/10Topics/Ver1.1/Train_Full"
TEST_DIR  = "/content/VNTC/Data/10Topics/Ver1.1/Test_Full"
BATCH_SIZE = 8
EPOCHS = 2
MAX_LENGTH = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 🔹 SEED
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

# =========================
# 🔹 VnCoreNLP
# =========================
vncorenlp_path = "VnCoreNLP-master"
jar_path = os.path.join(vncorenlp_path, "VnCoreNLP-1.1.1.jar")
rdrsegmenter = None
def get_segmenter():
    global rdrsegmenter
    if rdrsegmenter is None:
        rdrsegmenter = VnCoreNLP(jar_path, annotators="wseg", max_heap_size='-Xmx2g')
    return rdrsegmenter

def segment_text(text):
    try:
        seg = get_segmenter()
        sents = seg.tokenize(text)
        return " ".join([" ".join(s) for s in sents])
    except:
        return ""

def load_or_segment(texts):
    return [segment_text(t) for t in tqdm(texts)]

# =========================
# 🔹 LOAD DATA
# =========================
def read_file_safe(path):
    for enc in ["utf-8", "utf-16", "latin-1"]:
        try:
            with open(path, encoding=enc) as f:
                return f.read().strip()
        except:
            continue
    return None

def load_data(data_dir, max_per_class=700, label_map=None):
    texts, labels = [], []
    if label_map is None:
        label_map = {}
    for label in sorted(os.listdir(data_dir)):
        if label not in label_map:
            label_map[label] = len(label_map)
        folder = os.path.join(data_dir, label)
        count = 0
        for file in os.listdir(folder):
            if count >= max_per_class:
                break
            text = read_file_safe(os.path.join(folder, file))
            if not text:
                continue
            texts.append(text)
            labels.append(label_map[label])
            count += 1
    return texts, labels, label_map

# =========================
# 🔹 DATASET
# =========================
class PhoBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = load_or_segment(texts)
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k,v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# =========================
# 🔹 CLASS WEIGHTS
# =========================
def compute_class_weights(labels, num_classes):
    counter = Counter(labels)
    total = sum(counter.values())
    weights = []
    for i in range(num_classes):
        count = counter[i] if i in counter else 0
        weights.append(total / (num_classes*count) if count>0 else 0.0)
    return torch.tensor(weights, dtype=torch.float).to(device)

# =========================
# 🔹 EVALUATION PER CLASS
# =========================
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def evaluate_phobert(
    model,
    loader,
    num_classes,
    texts=None,
    label_map=None,
    show_errors=True,
    print_per_class=True
):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # =========================
    # ✅ Overall metrics
    # =========================
    overall_acc = accuracy_score(all_labels, all_preds)
    overall_f1 = f1_score(all_labels, all_preds, average="weighted")

    print(f"\n✅ PhoBERT Acc: {overall_acc:.4f} | F1: {overall_f1:.4f}")

    # =========================
    # 📊 Per-class metrics
    # =========================
    if print_per_class:
        print("\n📊 Classification Report:\n")
        print(classification_report(all_labels, all_preds, digits=4))
    # =========================
    # 🔥 Most Confused Pairs
    # =========================
    from collections import Counter

    if show_errors:
        pairs = Counter()

        for t, p in zip(all_labels, all_preds):
            if t != p:
                pairs[(t, p)] += 1

        print("\n🔥 Most confused pairs:")

        inv_map = None
        if label_map:
            inv_map = {v: k for k, v in label_map.items()}

        for (t, p), count in pairs.most_common(5):
            if inv_map:
                t = inv_map[t]
                p = inv_map[p]
            print(f"{t} → {p}: {count}")
    # =========================
    # 🔥 Confusion Matrix
    # =========================
    if show_errors:
        cm = confusion_matrix(all_labels, all_preds)
        print("\n🔥 Confusion Matrix:\n")
        print(cm)

    # =========================
    # 📈 Per-class accuracy/F1 (your original)
    # =========================
    acc_per_class = {}
    f1_per_class = {}

    for cls in range(num_classes):
        mask = all_labels == cls
        if mask.sum() == 0:
            acc_per_class[cls] = 0.0
            f1_per_class[cls] = 0.0
        else:
            acc_per_class[cls] = (all_preds[mask] == all_labels[mask]).mean()
            f1_per_class[cls] = f1_score(
                all_labels[mask],
                all_preds[mask],
                average="weighted"
            )

    # =========================
    # ❌ Error Analysis
    # =========================
    if show_errors and texts is not None:
        print("\n❌ Misclassified Examples:\n")

        mis_idx = np.where(all_preds != all_labels)[0]

        inv_map = None
        if label_map:
            inv_map = {v: k for k, v in label_map.items()}

        for i in mis_idx[:5]:
            true_label = all_labels[i]
            pred_label = all_preds[i]

            if inv_map:
                true_label = inv_map[true_label]
                pred_label = inv_map[pred_label]

            print(f"TEXT: {texts[i][:200]}...")
            print(f"TRUE: {true_label} | PRED: {pred_label}")
            print("-" * 60)

    return overall_acc, overall_f1, acc_per_class, f1_per_class

# =========================
# 🔹 BUILD DATALOADERS
# =========================
def build_dataloaders():
    train_texts, train_labels, label_map = load_data(TRAIN_DIR)
    test_texts, test_labels, _ = load_data(TEST_DIR, label_map=label_map)
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts, train_labels,
        test_size=0.2,
        stratify=train_labels,
        random_state=42
    )
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
    train_ds = PhoBERTDataset(train_texts, train_labels, tokenizer)
    val_ds   = PhoBERTDataset(val_texts, val_labels, tokenizer)
    test_ds  = PhoBERTDataset(test_texts, test_labels, tokenizer)
    return DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True), \
           DataLoader(val_ds, batch_size=BATCH_SIZE), \
           DataLoader(test_ds, batch_size=BATCH_SIZE), \
           len(label_map), train_labels, test_texts, label_map

# =========================
# 🔹 MODEL
# =========================
# =========================
# 🔹 MODEL (Updated for frozen base)
# =========================
class PhoBERTClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            "vinai/phobert-base",
            num_labels=num_classes
        )

        # 1. Freeze the base model (PhoBERT uses the RoBERTa architecture)
        for param in self.model.roberta.parameters():
            param.requires_grad = False

        # The classification head (self.model.classifier) remains unfrozen
        # by default, so it will be the only part that updates.

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

# =========================
# 🔹 TRAIN MODEL
# =========================
def train_model():
    train_loader, val_loader, test_loader, num_classes, train_labels, test_texts, label_map = build_dataloaders()
    model = PhoBERTClassifier(num_classes).to(device)

    class_weights = compute_class_weights(train_labels, num_classes)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-3)
    total_steps = len(train_loader)*EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)
    train_time = 0
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        model.train()
        total_loss = 0

        start_epoch = time.time()

        for batch in tqdm(train_loader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask, labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        train_time += time.time() - start_epoch
        print(f"Loss: {total_loss/len(train_loader):.4f}")

        # Validation
        val_acc, val_f1, _, _ = evaluate_phobert(
            model,
            val_loader,
            num_classes,
            show_errors=False,
            print_per_class=False
        )
        print(f"Val Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

    print(f"PhoBERT Training Time: {train_time:.2f}s")
    # Test
    start_test = time.time()
    test_acc, test_f1, _, _ = evaluate_phobert(
        model,
        test_loader,
        num_classes,
        show_errors=False,
        print_per_class=False
    )
    end_test = time.time()
    print(f"PhoBERT Inference Time: {end_test - start_test:.2f}s")
    print(f"\nTest Acc: {test_acc:.4f} | F1: {test_f1:.4f}")
    evaluate_phobert(
        model,
        test_loader,
        num_classes,
        texts=test_texts,
        label_map=label_map,
        show_errors=True,
        print_per_class=True
    )
# =========================
# 🔹 RUN
# =========================
train_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


Epoch 1/2



100%|██████████| 700/700 [48:50<00:00,  4.19s/it]


Loss: 0.8139

✅ PhoBERT Acc: 0.8329 | F1: 0.8346
Val Acc: 0.8329 | F1: 0.8346

Epoch 2/2


100%|██████████| 700/700 [48:45<00:00,  4.18s/it]


Loss: 0.4298

✅ PhoBERT Acc: 0.8629 | F1: 0.8626
Val Acc: 0.8629 | F1: 0.8626
PhoBERT Training Time: 5856.27s

✅ PhoBERT Acc: 0.8341 | F1: 0.8274
PhoBERT Inference Time: 2956.48s

Test Acc: 0.8341 | F1: 0.8274

✅ PhoBERT Acc: 0.8341 | F1: 0.8274

📊 Classification Report:

              precision    recall  f1-score   support

           0     0.7485    0.7014    0.7242       700
           1     0.8191    0.4529    0.5833       700
           2     0.7510    0.7757    0.7632       700
           3     0.8044    0.8929    0.8463       700
           4     0.8714    0.9200    0.8951       700
           5     0.8506    0.8786    0.8644       700
           6     0.8357    0.9371    0.8835       700
           7     0.9577    0.9714    0.9645       700
           8     0.8139    0.9186    0.8631       700
           9     0.8803    0.8929    0.8865       700

    accuracy                         0.8341      7000
   macro avg     0.8333    0.8341    0.8274      7000
weighted avg     0.8333